### Part 1 - AURA Archiving

AURA uses zip files to reduce the inode usage (number of files), however this creates some problems for users...

AURA zip files can be very large, sometimes exceeding 5 GB/radar/day for level 1 and level 1b files. QPE products also use daily zip files.

This means they take some time to unzip locally and require lots of disk space when unzipping many files.

This tutorial will work through some practical ways to deal with this.

Let's first start by defining a test dataset from AURA

In [ ]:
from datetime import datetime

base_url = 'https://thredds.nci.org.au/thredds/fileServer/rq0/' #base url for level 1 datasets
radar_id = 50 #radar id for the desired radar
date_str = '20141127' #date in YYYYMMDD format
start_time_str = '0700' #start date in HHMM format
end_time_str = '0800' #end date in HHMM format

# parse and define file path
start_dt     = datetime.strptime(f'{date_str}_{start_time_str}', '%Y%m%d_%H%M')
end_dt       = datetime.strptime(f'{date_str}_{end_time_str}', '%Y%m%d_%H%M')
base_url     = 'https://thredds.nci.org.au/thredds/fileServer/rq0/' #base url for NCI level 1 dataset
zip_fn       = f'{radar_id}_{start_dt.strftime("%Y%m%d")}.pvol.zip'
request_url  = '/'.join([base_url, str(radar_id), start_dt.strftime('%Y'), 'vol', zip_fn])

#this is the path of the file to download (approx 15 MB zip file located on NCI THREDDS server)
print(f'Request URL: {request_url}')

Request URL: https://thredds.nci.org.au/thredds/fileServer/rq0//50/2014/vol/50_20141127.pvol.zip


In [ ]:
# Since we're working locally or on colab we'll need to download this zip file to our machine.

# If you're running this code on NCI (and using the direct path for level 1 data: /g/data/rq0/level_1, then you can skip this).

def download_file(url, local_filename=None, overwrite=False, show_progress=True, chunk_size=8192):
    """
    Download URL to local_filename. Streams to disk and shows optional progress.
    Returns the path to the downloaded file (str).
    """
    if local_filename is None:
        local_filename = Path("/tmp") / Path(url).name
    local_filename = Path(local_filename)

    if local_filename.exists() and not overwrite:
        print(f"File already exists: {local_filename}")
        return str(local_filename)

    print(f"Downloading {url} -> {local_filename} ...")
    try:
        # Preferred: requests (streaming)
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            if show_progress and tqdm and total:
                pbar = tqdm(total=total, unit="B", unit_scale=True)
            else:
                pbar = None

            with local_filename.open("wb") as f:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if not chunk:
                        continue
                    f.write(chunk)
                    if pbar:
                        pbar.update(len(chunk))
            if pbar:
                pbar.close()
    except Exception as e:
        # Fallback to urllib (simpler)
        print(f"requests failed ({e}), falling back to urllib.request.urlretrieve")
        urllib.request.urlretrieve(url, str(local_filename))
    print(f"Downloaded: {local_filename}")
    return str(local_filename)

def list_zip_contents(zip_path):
    """
    List contents of zip file.
    """
    zip_path = Path(zip_path)
    if not zip_path.exists():
        raise FileNotFoundError(zip_path)

    with zipfile.ZipFile(zip_path, "r") as zf:
        file_list = sorted(zf.namelist())
    return file_list

def extract_zip_to_temp(zip_path):
    """
    Extract zip to a temporary directory and return the temp dir path and sorted file list.
    Caller is responsible for cleaning the temp dir (use shutil.rmtree).
    """
    zip_path = Path(zip_path)
    if not zip_path.exists():
        raise FileNotFoundError(zip_path)

    temp_dir = Path(tempfile.mkdtemp(prefix=f"{zip_path.stem}_"))
    print(f"Extracting {zip_path} -> {temp_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(path=temp_dir)

    file_list = sorted([str(p) for p in temp_dir.iterdir() if p.is_file()])
    return str(temp_dir), file_list

def get_date_from_filename(filename, delimiter='_', date_fmt='%Y%m%d_%H%M%S'):
    """
    INPUT:
    filename (str):
        can contain path, has no impact on this function.
        filename must use the convention IDDDD_DATE. delimiter + everything else
        DATE must have same format as date
    OUTPUT:
    radar_id (int)
    """
    if not isinstance(filename, str):
        raise ValueError(f"get_id_from_filename: filename is not a string: {filename}")
        return None
    if delimiter not in filename:
        raise ValueError(f"get_id_from_filename: Delimiter not found in filename: {filename}")
        return None
    fn = os.path.basename(filename)
    fn_parts = fn.split(delimiter)
    try:
        dtstr = fn_parts[1] + '_' + fn_parts[2].split('.')[0]
        dt = datetime.strptime(dtstr, date_fmt)
        return dt
    except:
        raise ValueError(f"get_id_from_filename: Failed to extract radar if from: {filename}")
        return None

In [ ]:
# Download (will skip if already exists)
zip_ffn_str = download_file(request_url, zip_ffn)

# Extract and list files; ensure cleanup afterwards
temp_dir = None
try:
    temp_dir, file_list = extract_zip_to_temp(zip_ffn_str)

    # parse datetimes from filenames
    file_dt_list = []
    for fname in file_list:
        try:
            file_dt_list.append(get_datetime_from_filename(fname))
        except ValueError as e:
            print("Warning:", e)
            file_dt_list.append(None)

    # filter files that fall within time window
    filter_file_list = [
        f
        for f, dt in zip(file_list, file_dt_list)
        if (dt is not None and dt >= start_dt and dt <= end_dt)
    ]

    print(f"Found {len(filter_file_list)} files in range {start_dt} -> {end_dt}")
    for f in filter_file_list:
        print("  ", f)

finally:
    # Clean up temporary extraction directory to save space
    if temp_dir and Path(temp_dir).exists():
        print("Cleaning up temporary directory:", temp_dir)
        shutil.rmtree(temp_dir)